# Step 1 — Build BERTopic Model

Edit the **configuration cell** below, then run all cells.
The trained model, annotated data, and run parameters are saved together under `output/<run_name>/`
so you can keep multiple models and load any of them in `02_analyze_patterns.ipynb`.

In [ ]:
# ============================================================
# CONFIGURATION — edit this cell, then run all cells
# ============================================================

# Embedding model:
#   "default"            -> all-MiniLM-L6-v2  (fast, general-purpose)
#   "clinical_radiology" -> Bio_ClinicalBERT   (domain-specific, slower)
EMBEDDING = "default"

# Stop-word list:
#   "english"             -> standard sklearn English list
#   "medical_extended"    -> English + common medical/admin terms
#   "radiology_stopwords" -> English + radiology/imaging terms
STOPWORDS = "english"

# Data source:
#   True  -> read from local CSV (mock data, development)
#   False -> connect via turbodbc (production database)
USE_LOCAL_DATA = True

# UMAP — reduce n_neighbors for small datasets
#   Mock data (~20 docs):    n_neighbors=5
#   Production (1000+ docs): n_neighbors=15
UMAP_N_NEIGHBORS = 5
UMAP_N_COMPONENTS = 5

# HDBSCAN — reduce min_cluster_size for small datasets
#   Mock data:   min_cluster_size=3, min_samples=2
#   Production:  min_cluster_size=10, min_samples=5
HDBSCAN_MIN_CLUSTER_SIZE = 3
HDBSCAN_MIN_SAMPLES = 2

# Optional custom name for this run.
# Leave empty to auto-generate from parameters, e.g. "default__english__20260604_143022".
RUN_TAG = ""

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import datetime
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

In [ ]:
if USE_LOCAL_DATA:
    print("Reading mock data from local CSV...")
    df = pd.read_csv("mock_clinical_data.csv")
else:
    import turbodbc
    conn_str = (
        "Driver={SQL Server};"
        "Server=your_server;"
        "Database=your_database;"
        "Trusted_Connection=yes;"
    )
    conn = turbodbc.connect(connection_string=conn_str)
    cursor = conn.cursor()
    cursor.execute("SELECT id, text, body_part, gender FROM clinical_records_table")
    rows = cursor.fetchall()
    df = pd.DataFrame(rows, columns=["id", "text", "body_part", "gender"])

print(f"Loaded {len(df)} records.")
display(df.head())

In [ ]:
# Embedding model
if EMBEDDING == "clinical_radiology":
    embedding_model = SentenceTransformer("emilyalsentzer/Bio_ClinicalBERT")
    print("Embedding: Bio_ClinicalBERT")
else:
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    print("Embedding: all-MiniLM-L6-v2")

# Stop-words
_base = list(CountVectorizer(stop_words="english").get_stop_words())
stopword_sets = {
    "english": _base,
    "medical_extended": list(set(_base) | {
        "patient", "patients", "hospital", "diagnosis", "medical", "report",
        "treatment", "clinic", "doctor", "physician", "nurse", "health", "care",
    }),
    "radiology_stopwords": list(set(_base) | {
        "imaging", "scan", "report", "findings", "radiology", "ct", "mri",
        "x-ray", "contrast", "study", "exam", "views", "series",
    }),
}
selected_stopwords = stopword_sets.get(STOPWORDS, stopword_sets["english"])
print(f"Stop-words: {STOPWORDS} ({len(selected_stopwords)} words)")
vectorizer_model = CountVectorizer(stop_words=selected_stopwords)

# Sub-models
umap_model = UMAP(
    n_neighbors=UMAP_N_NEIGHBORS, n_components=UMAP_N_COMPONENTS,
    min_dist=0.0, metric="cosine", random_state=42,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, min_samples=HDBSCAN_MIN_SAMPLES,
    metric="euclidean", prediction_data=True,
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=KeyBERTInspired(),
    verbose=True,
)
print("BERTopic configured.")

In [ ]:
docs = df["text"].tolist()
print(f"Fitting BERTopic on {len(docs)} documents...")
topics, probs = topic_model.fit_transform(docs)
df["topic"] = topics
print("Done.\n")

topic_info = topic_model.get_topic_info()
display(topic_info)

print("\nTop words per topic:")
for tid in sorted(set(topics)):
    words = topic_model.get_topic(tid)
    label = f"Topic {tid}" if tid != -1 else "Outliers (-1)"
    print(f"  {label}: {[w for w, _ in (words or [])[:8]]}")

In [ ]:
# Auto-generate run name from parameters + timestamp if no custom tag given
_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = RUN_TAG.strip() or f"{EMBEDDING}__{STOPWORDS}__{_ts}"
run_dir = os.path.join("output", run_name)
os.makedirs(run_dir, exist_ok=True)

# Save BERTopic model
model_path = os.path.join(run_dir, "model")
topic_model.save(model_path, serialization="safetensors", save_ctfidf=True)

# Save annotated data as Parquet
data_path = os.path.join(run_dir, "annotated_data.parquet")
df.to_parquet(data_path, index=False)

# Save run config + metadata so 02_analyze_patterns.ipynb can reload correctly
n_topics = len([t for t in set(topics) if t != -1])
run_config = {
    "run_name": run_name,
    "embedding": EMBEDDING,
    "stopwords": STOPWORDS,
    "use_local_data": USE_LOCAL_DATA,
    "umap_n_neighbors": UMAP_N_NEIGHBORS,
    "umap_n_components": UMAP_N_COMPONENTS,
    "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
    "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES,
    "trained_on": datetime.datetime.now().isoformat(timespec="seconds"),
    "n_docs": len(docs),
    "n_topics": n_topics,
}
with open(os.path.join(run_dir, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

print(f"Saved to: {run_dir}/")
print(f"  model/               <- BERTopic model")
print(f"  annotated_data.parquet")
print(f"  run_config.json      <- {n_topics} topics, {len(docs)} docs")
print(f"\nTo analyse: set MODEL_TO_LOAD = \"{run_name}\" in 02_analyze_patterns.ipynb")